# 12 — Regressão Linear, Múltipla e Regularização

**EP · Data Science Knowledge Base | Machine Learning | Notebook 12/17**

**Execução:** Reinicie o kernel e execute as células em ordem. Observe as instruções específicas e as dependências opcionais indicadas neste notebook.

**Dados e dependências:** exemplos com dados sintéticos ou datasets integrados ao scikit-learn, sem acesso à internet.

**Objetivo geral:** construir compreensão conceitual e prática para aplicar este tema em problemas de Machine Learning, com avaliação adequada e interpretação dos resultados.


## Materiais complementares

- [Guia Técnico](./reference_materials/12_regressao_linear.pdf)
- [Cheat Sheet](./cheat_sheets/cheat_sheet_regressao_linear.pdf)


## 1. Fundamentos, objetivos e contexto

**Objetivos.** Revisar equação de regressão (Área 03), aplicar validação preditiva, entender Ridge/Lasso/Elastic Net e interpretação de coeficientes.

Mínimos quadrados ajustam média condicional sob função de perda quadrática. Predição não exige necessariamente que todos os pressupostos inferenciais clássicos sejam verdadeiros, mas desvios afetam interpretação e incerteza. Regularização é útil com colinearidade e muitos atributos; os coeficientes dependem da escala.


## 2. Fundamentos matemáticos

Modelo $\hat y=\beta_0+X\beta$; OLS minimiza $\sum_i(y_i-\hat y_i)^2$. Ridge acrescenta $\alpha\lVert\beta\rVert_2^2$; Lasso acrescenta $\alpha\lVert\beta\rVert_1$; Elastic Net combina ambas. O intercepto geralmente não é penalizado.

**Conexões com áreas anteriores:** revise os conceitos matemáticos e estatísticos correspondentes. Não repetimos integralmente os capítulos de Python, visualização e inferência estatística.


## 3. Experimento guiado: preparando a pergunta

**Antes do código**, identifique: (a) quais dados são de treino, (b) qual informação poderá entrar no modelo, (c) que métrica ou propriedade será observada, e (d) que conclusão não é permitida pelos resultados. Estes exemplos são didáticos, com semente fixa onde aplicável.


In [1]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, r2_score
X, y = load_diabetes(return_X_y=True)
a, b, c, d = train_test_split(X, y, test_size=.25, random_state=42)
candidatos = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(alpha=1.),
    'Lasso': Lasso(alpha=.1, max_iter=10000),
    'ElasticNet': ElasticNet(alpha=.001, l1_ratio=.5, max_iter=10000),
}
cv = KFold(5, shuffle=True, random_state=42)
cv_mae = {}
for nome, reg in candidatos.items():
    estimador = make_pipeline(StandardScaler(), reg)
    cv_mae[nome] = -cross_val_score(
        estimador, a, c, cv=cv, scoring='neg_mean_absolute_error').mean()
    print(nome, 'MAE CV treino:', round(cv_mae[nome], 2))
vencedor = min(cv_mae, key=cv_mae.get)
final = make_pipeline(StandardScaler(), candidatos[vencedor]).fit(a, c)
prev = final.predict(b)
print('Modelo selecionado por CV:', vencedor)
print('Teste reservado: MAE', round(mean_absolute_error(d, prev), 2),
      'R²', round(r2_score(d, prev), 3))


LinearRegression MAE CV treino: 45.81
Ridge MAE CV treino: 45.75
Lasso MAE CV treino: 45.75
ElasticNet MAE CV treino: 45.79
Modelo selecionado por CV: Lasso
Teste reservado: MAE 41.48 R² 0.487


### Como interpretar

Os números são resultados específicos do experimento e da amostra, não garantias universais. Verifique o que mudou, qual método foi usado e qual é o cenário de aplicação. Compare com a teoria matemática e não confunda desempenho de treino, validação e teste.


## 4. Segundo experimento ou extensão controlada

Mude apenas uma condição principal (algoritmo, parâmetro ou tipo de dado), quando possível, e registre o efeito. Não ajuste decisões com base no teste final.


In [2]:
import numpy as np
from sklearn.linear_model import Ridge
for alpha in [.01,1,100]:
    m=make_pipeline(StandardScaler(),Ridge(alpha=alpha)).fit(a,c)
    print('alpha',alpha,'norma coeficientes',round(np.linalg.norm(m.named_steps['ridge'].coef_),2))

alpha 0.01 norma coeficientes 70.31
alpha 1 norma coeficientes 59.75
alpha 100 norma coeficientes 33.79


## 5. Exercício resolvido

**Enunciado:** reforce o conceito central com um exemplo menor, cujo resultado pode ser inspecionado diretamente.


In [3]:
# Exercício resolvido: previsão de uma regressão linear simples
intercepto,inclinacao=10.,2.5
for x in [0,4,8]:print('x:',x,'previsão:',intercepto+inclinacao*x)

x: 0 previsão: 10.0
x: 4 previsão: 20.0
x: 8 previsão: 30.0


## 6. Interpretação, limitações e erros comuns

R² pode ser negativo no teste. OLS não é apropriado como prova de causalidade. Regularização exige escolha de intensidade por validação, não por menor erro de treino.

**Perguntas de revisão:**

1. Qual hipótese implícita sustenta o procedimento usado?
2. Que erro de implementação ou de validação poderia tornar o resultado artificialmente otimista?
3. Como você mudaria o experimento para refletir dados reais?


## 7. Consulta rápida

- OLS otimiza MSE; Ridge usa L2; Lasso usa L1; Elastic Net combina.
- MAE é mais robusto a erros grandes do que RMSE.
- Validar com holdout apropriado e analisar resíduos.
- Coeficientes não equivalem a efeitos causais.


## 8. Exercícios adicionais (para fazer no JupyterLab)

**Exercício A — sensibilidade:** altere um hiperparâmetro/hipótese e apresente uma tabela pequena com resultados e explicação. **Exercício B — revisão crítica:** construa um exemplo de uso incorreto (sem aplicá-lo a dados reais) e explique o procedimento correto. **Exercício C — comunicação:** escreva um parágrafo sobre escopo, métricas, suposições e limitações do método.

**Critério de aprovação:** códigos executam do começo ao fim, não há seleção pelo teste final e cada resultado possui interpretação.


## 9. Fontes e continuidade curricular

**Origem principal das aulas:** 14-regressao.pdf; Regressão_Linear_Simples.ipynb; Regressão_Linear_Múltipla.ipynb; Área 03 — NB10.

**Referências de atualização:** documentação de `scikit-learn` (<https://scikit-learn.org/stable/>); explicações matemáticas redigidas para esta reconstrução. Outros pacotes (Optuna, XGBoost, SHAP, imbalanced-learn) não são necessários para executar este capítulo.



## Aprofundamento — auditoria e aplicação

**Ponto de atenção deste capítulo:** resíduos e intervalos. Este complemento não substitui as seções e exemplos anteriores.

**Verificação conceitual.** Explique, com suas palavras, em qual etapa do fluxo de trabalho o tema **resíduos e intervalos** pode produzir conclusões enganosas. Indique um indicador observável, um teste mínimo e uma limitação do procedimento.

**Exercício de extensão.** Reproduza o experimento central deste notebook alterando **um parâmetro por vez**. Compare com uma linha de base simples, reporte a variação da métrica pertinente e justifique o resultado. Quando houver treino/teste, qualquer etapa que aprenda parâmetros deve usar apenas o treino.

**Checklist de conclusão:** (1) entradas identificadas; (2) sementes registradas; (3) saídas interpretadas; (4) premissas explicitadas; (5) dados originais não sobrescritos.


### Exercício resolvido — protocolo mínimo de validação

**Enunciado.** Proponha uma comparação para **resíduos e intervalos** sem contaminar a avaliação.

**Solução orientadora.** Defina uma hipótese falsificável, escolha um baseline simples, fixe uma divisão adequada ao problema, altere uma única decisão metodológica e compare métricas no conjunto reservado. Não transforme exemplos sintéticos em conclusões sobre populações reais. Documente a configuração e os casos de falha. Quando o tema não envolver predição, substitua o baseline por uma implementação de referência e compare correção, cobertura e custo.
